## Imports and paths

In [1]:
from pathlib import Path
import unicodedata
import numpy as np
import pandas as pd
from IPython.display import display

In [2]:
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 100)

PROJECT_DIR = Path.cwd()
UNIFORM_DIR = PROJECT_DIR / "Cleaned Data" / "Uniform column format"

COUNTRIES = {"AUS": "Australia","BRA": "Brazil","CAN": "Canada","GBR": "United Kingdom","IND": "India","USA": "United States",}

## 1 Check each country's YouGov states against OxCGRT regions

### 1.1 Help function

In [3]:
# Read function
def read_csv_clean(path):
    return pd.read_csv(path,na_values=[" ", "__NA__"],keep_default_na=True,low_memory=False)

# Normalize text
def norm_text(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip().lower()
    x = unicodedata.normalize("NFKD", x)
    x = "".join(ch for ch in x if not unicodedata.combining(ch))
    x = x.replace("-", " ").replace("_", " ")
    x = " ".join(x.split())

    return x

# Remove accents
def remove_accents(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()
    x = unicodedata.normalize("NFKD", x)
    x = "".join(ch for ch in x if not unicodedata.combining(ch))
    x = " ".join(x.split())

    return x

# Find differences  between YouGov  and  OxCGRT
def check_unmatched_states(yougov_df, oxcgrt_df):
    yougov = pd.DataFrame({"yougov_state": sorted(yougov_df["state"].dropna().unique())})
    oxcgrt = pd.DataFrame({"oxcgrt_region": sorted(oxcgrt_df["RegionName"].dropna().unique())})

    yougov["key"] = yougov["yougov_state"].map(norm_text)
    oxcgrt["key"] = oxcgrt["oxcgrt_region"].map(norm_text)

    yougov_only = (yougov.loc[~yougov["key"].isin(set(oxcgrt["key"])), "yougov_state"].reset_index(drop=True).rename("yougov_state_without_oxcgrt_match"))
    oxcgrt_only = (oxcgrt.loc[~oxcgrt["key"].isin(set(yougov["key"])), "oxcgrt_region"].reset_index(drop=True).rename("oxcgrt_region_without_yougov_match"))

    return pd.concat([yougov_only, oxcgrt_only], axis=1)

### 1.2 Differences

In [4]:
before_unmatched_tables = {}

for code, country_name in COUNTRIES.items():
    print("\n" + "=" * 100)
    print(f"{country_name}")
    print("=" * 100)

    yougov_path = UNIFORM_DIR / f"YouGov_{country_name}.csv"
    oxcgrt_path = UNIFORM_DIR / f"OxCGRT_{country_name}.csv"

    yougov_df = read_csv_clean(yougov_path)
    oxcgrt_df = read_csv_clean(oxcgrt_path)

    unmatched_table = check_unmatched_states(yougov_df, oxcgrt_df)
    before_unmatched_tables[code] = unmatched_table

    if unmatched_table.empty:
        print("Everything ok")
    else:
        print("Error:")
        display(unmatched_table)


Australia
Everything ok

Brazil
Everything ok

Canada
Error:


,yougov_state_without_oxcgrt_match,oxcgrt_region_without_yougov_match
0,British Columbia / Colombie Britanique,British Columbia
1,New Brunswick / Nouveau-Brunswick,New Brunswick
2,Newfoundland & Labrador / Terre-Neuve-et-Labrador,Newfoundland and Labrador
3,Northwest Territories / Territoires du Nord-Ouest,Northwest Territories
4,Nova Scotia / Nouvelle-Écosse,Nova Scotia
5,Prince Edward Island / Île-du-Prince-Édouard,Prince Edward Island
6,Quebec / Québec,Quebec



United Kingdom
Error:


,yougov_state_without_oxcgrt_match,oxcgrt_region_without_yougov_match
0,East Midlands,England
1,East of England,NaN
2,London,NaN
3,North East,NaN
4,North West,NaN
5,South East,NaN
6,South West,NaN
7,West Midlands,NaN
8,Yorkshire and the Humber,NaN



India
Error:


,yougov_state_without_oxcgrt_match,oxcgrt_region_without_yougov_match
0,Andaman & Nicobar Islands,Andaman and Nicobar Islands
1,Dadra & Nagar Haveli,Dadra and Nagar Haveli and Daman and Diu
2,Daman & Diu,Jammu and Kashmir
3,Jammu & Kashmir,Ladakh



United States
Error:


,yougov_state_without_oxcgrt_match,oxcgrt_region_without_yougov_match
0,District of Columbia,Washington DC


## 2 Correct Errors

### 2.1 Regional names to be modified or deleted

In [5]:
STATE_MAPPING = {
    
    # Correct some naming errors (Canada)
    "Canada": {
        "British Columbia / Colombie Britanique": "British Columbia",
        "British Columbia / Colombie Britannique": "British Columbia",
        "New Brunswick / Nouveau-Brunswick": "New Brunswick",
        "Newfoundland & Labrador / Terre-Neuve-et-Labrador": "Newfoundland and Labrador",
        "Northwest Territories / Territoires du Nord-Ouest": "Northwest Territories",
        "Nova Scotia / Nouvelle-Écosse": "Nova Scotia",
        "Prince Edward Island / Île-du-Prince-Édouard": "Prince Edward Island",
        "Quebec / Québec": "Quebec"},

    # Correct some naming errors (UK)
    "United Kingdom": {
        "East Midlands": "England",
        "East of England": "England",
        "London": "England",
        "North East": "England",
        "North West": "England",
        "South East": "England",
        "South West": "England",
        "West Midlands": "England",
        "Yorkshire and the Humber": "England"},

    # Correct some naming errors (India)
    "India": {
        "Andaman & Nicobar Islands": "Andaman and Nicobar Islands",
        "Dadra & Nagar Haveli": "Dadra and Nagar Haveli and Daman and Diu",
        "Daman & Diu": "Dadra and Nagar Haveli and Daman and Diu",
        "Jammu & Kashmir": "Jammu and Kashmir",},

    # Correct some naming errors (US)
    "United States": {"District of Columbia": "Washington DC"}}


# Delete regions that cannot be mapped (India)
DROP_OXCGRT_REGIONS = {"India": ["Ladakh"]}

### 2.2 Update and Save

In [6]:
for country_name in COUNTRIES.values():
    yougov_path = UNIFORM_DIR / f"YouGov_{country_name}.csv"
    oxcgrt_path = UNIFORM_DIR / f"OxCGRT_{country_name}.csv"

    yougov_df = read_csv_clean(yougov_path)
    yougov_df["state"] = yougov_df["state"].replace(STATE_MAPPING.get(country_name, {}))
    yougov_df["state"] = yougov_df["state"].map(remove_accents)
    yougov_df.to_csv(yougov_path, index=False, encoding="utf-8-sig")

    oxcgrt_df = read_csv_clean(oxcgrt_path)
    oxcgrt_df = oxcgrt_df[~oxcgrt_df["RegionName"].isin(DROP_OXCGRT_REGIONS.get(country_name, []))]
    oxcgrt_df["RegionName"] = oxcgrt_df["RegionName"].map(remove_accents)
    oxcgrt_df.to_csv(oxcgrt_path, index=False, encoding="utf-8-sig")


## 3 Check again

In [7]:
# Store
final_unmatched = []

# Check whether every YouGov state exists in OxCGRT for each country
for country_name in COUNTRIES.values():
    yougov_path = UNIFORM_DIR / f"YouGov_{country_name}.csv"
    oxcgrt_path = UNIFORM_DIR / f"OxCGRT_{country_name}.csv"

    yougov_df = read_csv_clean(yougov_path)
    oxcgrt_df = read_csv_clean(oxcgrt_path)

    yougov_states = pd.DataFrame({"yougov_state": sorted(yougov_df["state"].dropna().unique())})
    oxcgrt_regions = pd.DataFrame({"oxcgrt_region": sorted(oxcgrt_df["RegionName"].dropna().unique())})

    yougov_states["key"] = yougov_states["yougov_state"].map(norm_text)
    oxcgrt_keys = set(oxcgrt_regions["oxcgrt_region"].map(norm_text).dropna())

    missing = yougov_states.loc[~yougov_states["key"].isin(oxcgrt_keys),["yougov_state"]].copy()

    if not missing.empty:
        missing.insert(0, "country", country_name)
        final_unmatched.append(missing)

if final_unmatched:
    display(pd.concat(final_unmatched, ignore_index=True))
else:
    print("All mappings are completed")

All mappings are completed
